In [ ]:
# from pydub import AudioSegment

# # merge audio
# audio = AudioSegment.empty()

# for i in range(1, 11):
#     path = f"../data/audio/poem1_versuri/vers{i}.wav"
#     audio += AudioSegment.from_wav(path)

# # Audio(audio.raw_data, rate=audio.frame_rate)
# audio

In [ ]:
from pydub import AudioSegment

audio = AudioSegment.from_file("../data/raw-audio/1. Sailing_to_byzantium_simona_final.wav")

# remove the intro
audio = audio[2500:]

audio

In [ ]:
a = audio[2500:3150]
a.export("../results/exploration/poem1_the_young.wav")


## stress marker

In [ ]:
from dataclasses import fields
from typing import Any

from adapt_eeg.auto_annotation.stress_identifier import (
    StressIdentifyParams,
    detect_stressed_syllables,
)
from adapt_eeg.auto_annotation.syllable_identifier import SyllableIdentifyParams

class AllParams(StressIdentifyParams, SyllableIdentifyParams):

    def __init__(self, **kwargs) -> None:
        stress_fields = {f.name for f in fields(StressIdentifyParams)}
        syllable_fields = {f.name for f in fields(SyllableIdentifyParams)}

        StressIdentifyParams.__init__(
            self,
            **{k: v for k, v in kwargs.items() if k in stress_fields},
        )

        SyllableIdentifyParams.__init__(
            self,
            **{k: v for k, v in kwargs.items() if k in syllable_fields},
        )


params = AllParams(
    intensity_floor_hz=50.0,
    silence_threshold_relative_db=-25.0,
    min_nucleus_prominence_db=1.0,
    pitch_range_hz=(75.0, 500.0),
    pitch_time_step_s=0.01,
    voicing_half_window_s=0.04,
    min_voiced_fraction=0.50,
    min_pause_duration_s=0.2,
    pitch_percentile=90.0,
    #### syllables -> stress
    prominence_neighbor_half_window_s=0.05,
    prominence_min_nucleus_distance_s=0.04,
    min_pitch_prominence_hz=2.0,
    min_intensity_prominence_db=5.0,
    duration_neighbor_radius=3,
    stress_weights=(0.5, 0.3, 0.2),
    stress_score_threshold=-0.28,
)

#### syllables

In [ ]:
from adapt_eeg.auto_annotation.syllable_identifier import (
    detect_nucleus_candidates,
    get_f0_contour,
    get_intensity_contour,
    get_silence_threshold_db,
    _load_audio,
)

sound, audio, samples, sample_rate = _load_audio(audio)

intensity = get_intensity_contour(
    sound,
    params.intensity_floor_hz,
)

f0 = get_f0_contour(
    sound,
    pitch_time_step_s=params.pitch_time_step_s,
    pitch_range_hz=params.pitch_range_hz,
)

silence_threshold_db = get_silence_threshold_db(
    intensity,
    params.silence_threshold_relative_db,
)

nucleus_candidates = detect_nucleus_candidates(
    intensity,
    silence_threshold_db,
    params.min_nucleus_prominence_db,
)
nucleus_candidates_times = [x.time for x in nucleus_candidates]

In [ ]:
start, end = 2.71, 3.17

target_nuclei = [n for n in nucleus_candidates if start <= n.time <= end]
for n in target_nuclei:
    print(n.time)

In [ ]:
x, y = round(start * 1000), round(end * 1000)

audio[x: y]

In [ ]:
import matplotlib.pyplot as plt

start_s = 19.77
end_s = 20.34

mask = (intensity.times >= start_s) & (intensity.times <= end_s)

plt.figure(figsize=(14, 5))

# Intensity contour
plt.plot(
    intensity.times[mask],
    intensity.values[mask],
    label="Intensity",
)

# Silence threshold
plt.axhline(
    silence_threshold_db,
    linestyle="--",
    label=f"Silence threshold ({silence_threshold_db:.1f} dB)",
)

# Detected syllables
for n in nucleus_candidates:
    if n.time < start_s or n.time > end_s:
        continue

    plt.scatter(
        n.time,
        n.intensity_db,
        marker="x",
        s=70,
    )

plt.xlabel("Time (s)")
plt.ylabel("Intensity (dB)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
from adapt_eeg.auto_annotation.syllable_identifier import filter_voiced_nuclei

nuclei = filter_voiced_nuclei(
    nucleus_candidates,
    f0,
    params.voicing_half_window_s,
    params.min_voiced_fraction,
)

nuclei_times = [x.time for x in nuclei]

In [ ]:
from adapt_eeg.auto_annotation.syllable_identifier import (
    estimate_syllable_intervals,
    measure_syllables,
)

intervals = estimate_syllable_intervals(
    nuclei,
    intensity,
    sound.duration,
    params.min_pause_duration_s,
    silence_threshold_db,
)

syllables = measure_syllables(
    intervals,
    audio,
    samples,
    sample_rate,
    f0,
    params.pitch_percentile,
)

syllable_times = [x.nucleus for x in syllables]
len(syllables)

In [ ]:
from adapt_eeg.auto_annotation.syllable_identifier import Syllable

def find_syllable_idx_at_nucleus(
    syllables: list[Syllable],
    x: float,
) -> int | None:
    target = round(x, 2)

    left = 0
    right = len(syllables) - 1

    while left <= right:
        mid = (left + right) // 2
        nucleus = round(syllables[mid].nucleus, 2)

        if nucleus == target:
            return mid

        if nucleus < target:
            left = mid + 1
        else:
            right = mid - 1

    return None

#### Inspect long syllables

In [ ]:
from matplotlib import pyplot as plt


plt.hist([syllable.duration_s for syllable in syllables if syllable.duration_s >= 0.6])
plt.show()

In [ ]:
long_syllables = [syllable for syllable in syllables if syllable.duration_s >= 0.6]
long_syllables[2]._audio  # monument

In [ ]:
long_syllables[2].start

In [ ]:
long_syllables[4]._audio  # sing outloud 

In [ ]:
long_syllables[4].start

In [ ]:
long_syllables[6]._audio  ## too low volumn, ignore

In [ ]:
long_syllables[7]._audio  ## too low volumn, ignore

In [ ]:
long_syllables[8]._audio  ## just sing

In [ ]:
find_syllable_idx_at_nucleus(syllables, )

#### stress

In [ ]:
from adapt_eeg.auto_annotation.stress_identifier import (
    _normalize_weights,
    calculate_stress_evidence,
    calculate_stress_scores,
    normalize_stress_evidence,
)
from adapt_eeg.auto_annotation.syllable_identifier import SyllablizedAudio

# f0_stress = get_f0_contour(
#     sound, params.pitch_time_step_s, (165, 255)
# )

syllablized_audio = SyllablizedAudio(
    sound=sound,
    audio=audio,
    samples=samples,
    sample_rate=sample_rate,
    intensity=intensity,
    f0=f0,
    silence_threshold_db=silence_threshold_db,
    syllables=syllables,
)

evidence = calculate_stress_evidence(
    syllablized_audio,
    params.prominence_neighbor_half_window_s,
    params.prominence_min_nucleus_distance_s,
    duration_neighbor_radius=params.duration_neighbor_radius,
    pitch_floor_hz=params.pitch_range_hz[0],
)

normalized_evidence = normalize_stress_evidence(evidence)

weights = _normalize_weights(params.stress_weights)

scores = calculate_stress_scores(
    normalized_evidence,
    weights,
)

stress_flags = classify_stresses(
    evidence,
    scores,
    params.min_pitch_prominence_hz,
    params.min_intensity_prominence_db,
    params.stress_score_threshold,
)

stresses = [syllable for syllable, stressed in zip(syllables, stress_flags) if stressed]
stress_times = [stress.nucleus for stress in stresses]

In [ ]:
for i, (syllable_time, score, flag) in enumerate(zip(evidence, scores, stress_flags)):
    print(syllable_time, score, flag)
    if i > 10:
        break

In [ ]:
for i, (syllable_time, score, flag) in enumerate(zip(normalized_evidence, scores, stress_flags)):
    print(syllable_time, score, flag)
    if i > 10:
        break

In [ ]:
syllables[1], scores[1]

In [ ]:
# prominences[0]

In [ ]:
# prominences[1]

## visualize

In [ ]:
from adapt_eeg.utils.audio_stress_visualization import waveform_render
from IPython.display import display, HTML


def waveform_player(audio, syllable_times, window_seconds):
    html = waveform_render(audio, syllable_times, window_seconds)
    display(HTML(html))

In [ ]:
waveform_player(audio, syllable_times, window_seconds=10)

In [ ]:
id = find_syllable_idx_at_nucleus(syllables, 2.92)
syllables[id]

In [ ]:
waveform_player(audio, stress_times, window_seconds=10)

In [ ]:
expected_stresses = [syllable_times[i] for i in [1,3,4,6,7] ]
waveform_player(audio, expected_stresses, window_seconds=10)

In [ ]:
syllable = syllables[1]
waveform_player(audio[:800], [syllable.start, syllable.nucleus, syllable.end])

In [ ]:
waveform_player(audio[:800], stress_times)

In [ ]:
import numpy as np

pitch_prom = np.array([e[0] for e in evidence], dtype=float)
intensity_prom = np.array([e[1] for e in evidence], dtype=float)

pitch_prom = pitch_prom[np.isfinite(pitch_prom)]
intensity_prom = intensity_prom[np.isfinite(intensity_prom)]

pitch_80_threshold = np.percentile(pitch_prom, 20)
intensity_80_threshold = np.percentile(intensity_prom, 20)

print("pitch threshold for ~80% pass:", pitch_80_threshold)
print("intensity threshold for ~80% pass:", intensity_80_threshold)

In [ ]:
print("syllables:", len(syllables))
print("stressed:", len(stresses))
print("stress %:", 100 * len(stresses) / len(syllables))

In [ ]:
for threshold in [0.50, 0.40, 0.30, 0.20, 0.10, 0.00]:
    # flags = [
    #     (score >= threshold and pitch_prom > 0 and intensity_prom > 0)
    #     for (pitch_prom, intensity_prom, _), score in zip(evidence, scores)
    # ]

    flags = classify_stresses(evidence, scores, 0,0,threshold)

    n = sum(flags)
    print(f"{threshold:.2f}: " f"{n}/{len(flags)} " f"({100*n/len(flags):.1f}%)")
    

In [ ]:
pitch_pass = [e[0] >= params.min_pitch_prominence_hz for e in evidence]

intensity_pass = [
    e[1] >= params.min_intensity_prominence_db for e in evidence
]

score_pass = [score >= params.stress_score_threshold for score in scores]

n = len(syllables)

print("pitch:", sum(pitch_pass), f"({100 * sum(pitch_pass) / n:.1f}%)")

print("intensity:", sum(intensity_pass), f"({100 * sum(intensity_pass) / n:.1f}%)")

print(
    "pitch + intensity:",
    sum(p and i for p, i in zip(pitch_pass, intensity_pass)),
    f"({100 * sum(p and i for p, i in zip(pitch_pass, intensity_pass)) / n:.1f}%)",
)

print("score:", sum(score_pass), f"({100 * sum(score_pass) / n:.1f}%)")

print(
    "all:",
    sum(
        p and i and s
        for p, i, s in zip(
            pitch_pass,
            intensity_pass,
            score_pass,
        )
    ),
)